In [1]:
# NLP Assignment 5: Token Classification (POS Tagging & Chunking)
### Fine-Tuning BERT using Hugging Face
## Step 1: Install & Import Libraries

!pip install transformers datasets seqeval
!pip install transformers[torch]
!pip install torch
!pip install ipywidgets
!pip install torch ipywidgets
!pip install -U datasets

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report

## Step 2: Load Dataset (CoNLL-2003)

dataset = load_dataset("conll2003", revision="refs/convert/parquet")

dataset["train"] = dataset["train"].select(range(500))
dataset["validation"] = dataset["validation"].select(range(200))

print(dataset)

## Step 3: Load Tokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

## Step 4: Tokenization & Label Alignment

label_list = dataset["train"].features["chunk_tags"].feature.names

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(example["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


## Step 5: Model Setup

model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased',num_labels=len(label_list))

## Step 6: Training

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator 
)

trainer.train()

## Step 7: Evaluation

predictions, labels, _ = trainer.predict(tokenized_datasets['validation'])
preds = np.argmax(predictions, axis=2)

true_labels = [[label_list[l] for l in label if l != -100] for label in labels]
true_preds = [[label_list[p] for (p, l) in zip(pred, label) if l != -100]
              for pred, label in zip(preds, labels)]

print(classification_report(true_labels, true_preds))

## Step 8: Inference

sentence = "John works at Google in California"

tokens = sentence.split()
inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs)
predictions = np.argmax(outputs.logits.detach().numpy(), axis=2)[0]

for token, pred in zip(tokens, predictions):
    print(f"{token} → {label_list[pred]}")

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

C:\Users\thrup\anaconda3.exe\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\thrup\.cache\huggingface\hub\datasets--conll2003. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\thrup\anaconda3.exe\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

        ADJP       0.00      0.00      0.00        11
        ADVP       0.00      0.00      0.00        24
        INTJ       0.00      0.00      0.00         1
          NP       0.56      0.73      0.63       777
          PP       0.79      0.76      0.78       194
         PRT       0.00      0.00      0.00        14
        SBAR       0.00      0.00      0.00        16
          VP       0.55      0.34      0.42       208

   micro avg       0.59      0.63      0.61      1245
   macro avg       0.24      0.23      0.23      1245
weighted avg       0.56      0.63      0.59      1245

John → O
works → B-NP
at → B-NP
Google → B-PP
in → I-NP
California → B-PP


C:\Users\thrup\anaconda3.exe\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
